# Figure 4: fusion-transcript landscape

This notebook contains the cleaned and reproducible workflow used to summarize
protein-coding fusion transcripts across tumor tissues.

## Analyses

- Figure 4b: number of fusion transcripts per sample
- Figure 4c: recurrent fusion transcripts shared across tissues
- Figure 4e: recurrent fusion-partner genes
- Figure 4f: classification of known and novel fusion events

All execution outputs, temporary inspections, duplicated analyses, and unrelated
database-preparation steps have been removed.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from upsetplot import UpSet, from_contents

sns.set_theme(style="white")


## Configuration


In [ ]:
project_dir = Path("/data1/HOMO_PANGENOME/DYY/data")
fusion_dir = project_dir / "fusion_gene"
output_dir = fusion_dir / "figure_4"
output_dir.mkdir(parents=True, exist_ok=True)

metadata_file = project_dir / "group_288transcript.csv"
fusion_file = fusion_dir / "1122protein_coding_fusion_gene.csv"
tumor_fusion_database_file = fusion_dir / "tumorfusion_database.tsv"

minimum_recurrent_samples = 2
minimum_shared_tissues = 2
top_partner_genes = 25


## Utility functions


In [ ]:
def require_file(file_path):
    """Raise an informative error when an input file is missing."""
    if not file_path.is_file():
        raise FileNotFoundError(f"Required file was not found: {file_path}")


def require_columns(table, required_columns, table_name):
    """Validate required table columns."""
    missing_columns = sorted(set(required_columns) - set(table.columns))

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def split_fusion_pair(fusion_series):
    """Split 'GENE1:GENE2' identifiers into two partner-gene columns."""
    split_values = fusion_series.astype(str).str.split(
        ":",
        n=1,
        expand=True,
    )

    if split_values.shape[1] != 2:
        raise ValueError(
            "Fusion identifiers must follow the 'GENE1:GENE2' format."
        )

    split_values.columns = ["gene_1", "gene_2"]
    return split_values


def save_figure(figure, file_name, dpi=400):
    """Save figures using consistent publication settings."""
    figure.savefig(
        output_dir / file_name,
        dpi=dpi,
        bbox_inches="tight",
    )


## Load and validate input data


In [ ]:
require_file(metadata_file)
require_file(fusion_file)

sample_metadata = pd.read_csv(metadata_file, sep="\t")
fusion_events = pd.read_csv(fusion_file, sep="\t")

require_columns(
    sample_metadata,
    {"name", "Sample", "Sample ID"},
    "Sample metadata",
)

require_columns(
    fusion_events,
    {"fusion genes", "sample", "tissue"},
    "Fusion-event table",
)

sample_metadata = sample_metadata.copy()
fusion_events = fusion_events.copy()

sample_metadata = sample_metadata.drop_duplicates("name")
fusion_events = fusion_events.drop_duplicates()

if fusion_events["sample"].isna().any():
    raise ValueError("The fusion-event table contains missing sample identifiers.")

if fusion_events["fusion genes"].isna().any():
    raise ValueError("The fusion-event table contains missing fusion identifiers.")


## Standardize fusion identifiers

Fusion pairs are represented as `GENE1:GENE2`. Partner genes are extracted explicitly
for downstream recurrence and novelty analyses.


In [ ]:
fusion_partners = split_fusion_pair(
    fusion_events["fusion genes"]
)

fusion_events = pd.concat(
    [
        fusion_events.reset_index(drop=True),
        fusion_partners.reset_index(drop=True),
    ],
    axis=1,
)

fusion_events["fusion_id"] = (
    fusion_events["gene_1"].astype(str)
    + ":"
    + fusion_events["gene_2"].astype(str)
)

fusion_events = fusion_events.drop_duplicates(
    subset=["sample", "fusion_id"]
)


# Figure 4b: fusion transcripts per sample

Each fusion event is counted once per sample. Samples are ordered by tissue and sample
identifier. Samples without retained fusion events are included with a count of zero
when they are present in the metadata.


In [ ]:
fusion_counts_per_sample = (
    fusion_events.groupby("sample")["fusion_id"]
    .nunique()
    .rename("fusion_count")
    .reset_index()
)

sample_plot_data = (
    sample_metadata[["name", "Sample", "Sample ID"]]
    .rename(columns={"name": "sample", "Sample": "tissue"})
    .merge(
        fusion_counts_per_sample,
        on="sample",
        how="left",
        validate="one_to_one",
    )
)

sample_plot_data["fusion_count"] = (
    sample_plot_data["fusion_count"]
    .fillna(0)
    .astype(int)
)

sample_plot_data = sample_plot_data.sort_values(
    ["tissue", "Sample ID", "sample"]
)

sample_plot_data.to_csv(
    output_dir / "figure_4b_fusion_counts_per_sample.tsv",
    sep="\t",
    index=False,
)


In [ ]:
figure_width = max(8, 0.16 * len(sample_plot_data))
figure, axis = plt.subplots(figsize=(figure_width, 4))

sns.barplot(
    data=sample_plot_data,
    x="sample",
    y="fusion_count",
    hue="tissue",
    dodge=False,
    ax=axis,
)

axis.axhline(
    sample_plot_data["fusion_count"].mean(),
    linestyle="--",
    linewidth=1,
)

axis.set_xlabel("Sample")
axis.set_ylabel("Number of fusion transcripts")
axis.tick_params(axis="x", rotation=90)
axis.legend(
    title="Tissue",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)
sns.despine(ax=axis)

save_figure(
    figure,
    "figure_4b_fusion_transcripts_per_sample.pdf",
)
plt.show()


# Figure 4c: recurrent fusion transcripts across tissues

A fusion is considered recurrent when it occurs in at least
`minimum_recurrent_samples` unique samples. Tissue-level membership is then represented
with an UpSet plot.


In [ ]:
fusion_recurrence = (
    fusion_events.groupby("fusion_id")
    .agg(
        sample_count=("sample", "nunique"),
        tissue_count=("tissue", "nunique"),
    )
    .sort_values(
        ["sample_count", "tissue_count"],
        ascending=False,
    )
)

recurrent_fusion_ids = fusion_recurrence.index[
    fusion_recurrence["sample_count"]
    >= minimum_recurrent_samples
]

recurrent_fusions = fusion_events.loc[
    fusion_events["fusion_id"].isin(recurrent_fusion_ids)
].copy()

recurrent_fusion_matrix = (
    recurrent_fusions.assign(present=1)
    .pivot_table(
        index="fusion_id",
        columns="tissue",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .astype(int)
)

recurrent_fusion_matrix.to_csv(
    output_dir / "figure_4c_recurrent_fusion_matrix.tsv",
    sep="\t",
)


In [ ]:
tissue_fusion_sets = {
    tissue: set(
        recurrent_fusion_matrix.index[
            recurrent_fusion_matrix[tissue].eq(1)
        ]
    )
    for tissue in recurrent_fusion_matrix.columns
}

tissue_fusion_sets = {
    tissue: fusion_ids
    for tissue, fusion_ids in tissue_fusion_sets.items()
    if fusion_ids
}

if len(tissue_fusion_sets) < 2:
    raise ValueError(
        "At least two tissues with recurrent fusion events are required "
        "for the UpSet plot."
    )

upset_data = from_contents(tissue_fusion_sets)

upset = UpSet(
    upset_data,
    min_degree=minimum_shared_tissues,
    sort_by="cardinality",
    show_counts=True,
    element_size=22,
)

upset.plot()
current_figure = plt.gcf()
save_figure(
    current_figure,
    "figure_4c_recurrent_fusion_upset.pdf",
)
plt.show()


# Figure 4e: recurrent fusion-partner genes

Both partners of every fusion event are combined, and the number of unique samples
containing each partner gene is calculated. The most recurrent genes are displayed.


In [ ]:
partner_gene_table = pd.concat(
    [
        fusion_events[["sample", "tissue", "gene_1"]]
        .rename(columns={"gene_1": "partner_gene"}),
        fusion_events[["sample", "tissue", "gene_2"]]
        .rename(columns={"gene_2": "partner_gene"}),
    ],
    ignore_index=True,
)

partner_gene_table = partner_gene_table.drop_duplicates(
    ["sample", "partner_gene"]
)

partner_gene_counts = (
    partner_gene_table.groupby("partner_gene")
    .agg(
        sample_count=("sample", "nunique"),
        tissue_count=("tissue", "nunique"),
    )
    .sort_values(
        ["sample_count", "tissue_count"],
        ascending=False,
    )
)

partner_gene_counts.to_csv(
    output_dir / "figure_4e_partner_gene_counts.tsv",
    sep="\t",
)


In [ ]:
top_partner_gene_counts = (
    partner_gene_counts.head(top_partner_genes)
    .sort_values("sample_count")
)

figure_height = max(
    4,
    0.25 * len(top_partner_gene_counts),
)

figure, axis = plt.subplots(
    figsize=(5, figure_height)
)

axis.barh(
    top_partner_gene_counts.index,
    top_partner_gene_counts["sample_count"],
)

axis.set_xlabel("Number of samples")
axis.set_ylabel("Fusion-partner gene")
sns.despine(ax=axis)

save_figure(
    figure,
    "figure_4e_recurrent_partner_genes.pdf",
)
plt.show()


# Figure 4f: known and novel fusion-event classes

Fusion events are classified using two complementary sources when available:

- `known == "Yes"` in the JAFFA-derived result table;
- exact fusion-pair or individual partner-gene matches in the external tumor-fusion
  database.

Events absent from both sources are classified as novel.


In [ ]:
unique_fusions = fusion_events.drop_duplicates(
    "fusion_id"
).copy()

unique_fusions["jaffa_known"] = False

if "known" in unique_fusions.columns:
    unique_fusions["jaffa_known"] = (
        unique_fusions["known"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
    )

unique_fusions["database_exact_match"] = False
unique_fusions["database_partner_match"] = False

if tumor_fusion_database_file.is_file():
    tumor_fusion_database = pd.read_csv(
        tumor_fusion_database_file,
        sep="\t",
    )

    require_columns(
        tumor_fusion_database,
        {"Gene_A", "Gene_B"},
        "Tumor-fusion database",
    )

    database_pairs = set(
        tumor_fusion_database["Gene_A"].astype(str)
        + ":"
        + tumor_fusion_database["Gene_B"].astype(str)
    )

    database_reverse_pairs = set(
        tumor_fusion_database["Gene_B"].astype(str)
        + ":"
        + tumor_fusion_database["Gene_A"].astype(str)
    )

    database_genes = set(
        tumor_fusion_database["Gene_A"].astype(str)
    ) | set(
        tumor_fusion_database["Gene_B"].astype(str)
    )

    unique_fusions["database_exact_match"] = (
        unique_fusions["fusion_id"].isin(
            database_pairs | database_reverse_pairs
        )
    )

    unique_fusions["database_partner_match"] = (
        unique_fusions["gene_1"].isin(database_genes)
        | unique_fusions["gene_2"].isin(database_genes)
    )


In [ ]:
classification_conditions = [
    unique_fusions["jaffa_known"]
    | unique_fusions["database_exact_match"],
    unique_fusions["database_partner_match"],
]

classification_labels = [
    "Known fusion pair",
    "Known partner gene only",
]

unique_fusions["fusion_class"] = np.select(
    classification_conditions,
    classification_labels,
    default="Novel fusion",
)

fusion_class_counts = (
    unique_fusions["fusion_class"]
    .value_counts()
    .rename_axis("fusion_class")
    .reset_index(name="fusion_count")
)

fusion_class_counts.to_csv(
    output_dir / "figure_4f_fusion_class_counts.tsv",
    sep="\t",
    index=False,
)

unique_fusions.to_csv(
    output_dir / "figure_4f_fusion_classification.tsv",
    sep="\t",
    index=False,
)


In [ ]:
figure, axis = plt.subplots(figsize=(4, 4))

axis.pie(
    fusion_class_counts["fusion_count"],
    labels=fusion_class_counts["fusion_class"],
    autopct="%1.1f%%",
    startangle=90,
)

axis.set_title("Fusion-event classification")
axis.axis("equal")

save_figure(
    figure,
    "figure_4f_fusion_classification.pdf",
)
plt.show()


## Reproducibility notes

- Notebook execution counters and outputs were cleared.
- Sample and fusion identifiers are aligned explicitly.
- Fusion events are deduplicated at the sample–fusion level.
- Recurrent-event counts use unique samples rather than raw row counts.
- Data conversion, Sanger-sequencing checks, one-off database inspections, and
  temporary filtering experiments from the original notebook are outside the scope of
  this release notebook and were removed.
